In [ ]:
import os
import re
import json
import time

from pathlib import Path
from collections import defaultdict

from semanticher.data import load_table_list, load_tables
from semanticher.onto import OntologyLoader, InversePropertyManager

from langchain_openai.chat_models.base import BaseChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# =========================
# Base folders
# =========================
REPO_ROOT       = Path().resolve().parents[1]
BASE_RESULT_DIR = REPO_ROOT / "results" / "main_msv_cnp"
CANDIDATE_DIR   = BASE_RESULT_DIR / "Result_msv_cnp" / "Candidate_msv_cnp"
LOG_DIR         = BASE_RESULT_DIR / "Promptinput_msv_cnp"
SUMMARY_DIR     = BASE_RESULT_DIR / "Tokenusage_Duration_msv_cnp"

os.makedirs(CANDIDATE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,       exist_ok=True)
os.makedirs(SUMMARY_DIR,   exist_ok=True)


# =========================
# Data / ontology
# =========================
ONTO_PATH    = REPO_ROOT / "data" / "ontology" / "BED.owl"
MAPPING_PATH = REPO_ROOT / "data" / "ontology" / "ambiguous_mapping.json"

loader      = OntologyLoader(str(ONTO_PATH), only_local=False, mapping_path=str(MAPPING_PATH))
ontology_json = loader.build()
print(loader.stats())

inv_mgr = InversePropertyManager(rdf_path=str(ONTO_PATH), only_local=False, mapping_path=str(MAPPING_PATH))
pairs, inverse_map = inv_mgr.prepare_inverse_conflicts()
print(f"[Inverse] pairs: {len(pairs)}")

# ambiguous mapping: name -> list of candidates with uri/comment/parents
with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    _amb_data = json.load(f)

AMBIGUOUS_CLASS_MAP = {}   # base_label -> [{"name":..,"uri":..,"comment":..,"parents":[..]}]
for entry in _amb_data.get("classes", []):
    label = entry.get("label", "").strip()
    if label:
        AMBIGUOUS_CLASS_MAP.setdefault(label, []).append(entry)

df_table_list = load_table_list()
dict_tables   = load_tables()

k = 5


def dataframe_to_markdown(df, k=5):
    subset  = df.head(k)
    headers = "| " + " | ".join(subset.columns) + " |"
    sep     = "| " + " | ".join(["---"] * len(subset.columns)) + " |"

    rows = []
    for _, row in subset.iterrows():
        row_str = "| " + " | ".join(map(str, row.values)) + " |"
        rows.append(row_str)

    return headers + "\n" + sep + "\n" + "\n".join(rows)


# =========================
# Model aliases / experiments
# =========================
MODEL_ALIAS = {
    "deepseek-chat":    "dsk",
    "gemini-2.0-flash": "gem",
    "gpt-4.1-mini":     "gpt",
}

EXPERIMENTS = [
    ["deepseek-chat"],
    ["gemini-2.0-flash"],
    ["gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash"],
    ["deepseek-chat", "gpt-4.1-mini"],
    ["gemini-2.0-flash", "gpt-4.1-mini"],
    ["deepseek-chat", "gemini-2.0-flash", "gpt-4.1-mini"],
]


def build_experiment_tag(model_names):
    aliases = [MODEL_ALIAS[m] for m in model_names]
    prefix  = {1: "single", 2: "two", 3: "three"}[len(model_names)]
    return f"{prefix}_{'_'.join(aliases)}"


def build_candidate_path(model_names):
    tag = build_experiment_tag(model_names)
    return CANDIDATE_DIR / f"candidate_msv_cnp_{tag}.json"


def build_logs_path(model_names):
    tag = build_experiment_tag(model_names)
    return LOG_DIR / f"msv_cnp_{tag}_logs.json"


def build_summary_path(model_names):
    tag = build_experiment_tag(model_names)
    return SUMMARY_DIR / f"summary_msv_cnp_{tag}.json"

In [ ]:
# =========================
# LLMs
# =========================
llm1 = BaseChatOpenAI(
    model="deepseek-chat",
    openai_api_key="",
    openai_api_base="https://api.deepseek.com",
    temperature=0,
    max_tokens=8192,
)

llm2 = BaseChatOpenAI(
    model="gemini-2.0-flash",
    openai_api_key="",
    openai_api_base="https://generativelanguage.googleapis.com/v1beta/openai",
    temperature=0,
    max_tokens=None,
)

llm3 = BaseChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key="",
    openai_api_base="https://api.openai.com/v1",
    temperature=0,
    max_tokens=None,
)

annotation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a knowledgeable assistant who helps map tabular data columns to ontology elements. You have expert knowledge in semantic table annotation. Your task is to map all columns of a given table to multiple plausible ontology classes and multiple plausible ontology properties. You must strictly select from the provided ontology lists only."
        ),
        (
            "human",
            """
            We have a table named "{table_name}" with several columns.
            Below is the table in Markdown format, including column headers and a few example rows:

            {table_in_markdown}

            Below is an ontology provided as a JSON object.

            Ontology:
            - The ontology contains two lists:
                - "classes": a list of class identifiers (strings)
                - "properties": a list of property identifiers (strings)
            - You may ONLY use identifiers that appear EXACTLY in these lists.
            - Do NOT invent new identifiers.
            - Identifiers are case-sensitive.
            - The ontology JSON is the only source of valid identifiers.

            Ontology (JSON):
            {ontology_json}

            Your task:
            For EACH column in the table, independently select:
            1) Top-k plausible class candidates
            2) Top-k plausible property candidates

            Guidelines:
            - Analyze all columns jointly.
            - Use column names, sample values, data types, and units as evidence.
            - Class candidates answer: "What type of concept is this column about?"
            - Property candidates answer: "What attribute or relation does this column represent?"
            - For measurement-like columns, it is common (but not required) to output:
                - a class such as Commodity / KPIValue / Time
                - a property such as hasValue and optionally hasUnit
            - If no suitable candidates exist for a list, output an empty list [].
            - Prefer semantic plausibility over being overly conservative.

            Active vs. passive semantics constraint:
            When selecting properties, you MUST distinguish between active (relational) properties and passive (attribute-like) properties:
            - Use hasX-style properties (e.g. hasName, hasDescription, hasValue) ONLY if the column represents a relation where the subject has or is linked to another entity or value.
            - Use plain attribute-like properties (e.g. name, description, value) ONLY if the column directly represents the attribute itself, not the act of having it.
            In other words:
            - If the column answers "what does this entity have?", choose hasX.
            - If the column answers "what is the X itself?", choose X.
            You MUST NOT treat hasX and X as interchangeable.

            Output format:
            - Output ONLY a valid JSON array.
            - Do not include any explanatory text.
            - Each array item corresponds to one input column, in the same order as the table.

            Each item must have the following structure:
            {{
            "column": "<exact column header>",
            "class": [["<class_name>", <confidence>], ...],
            "property": [["<property_name>", <confidence>], ...]
            }}

            Notes:
            - "class" and "property" are lists of pairs.
            - Each pair is ["identifier", confidence].
            - confidence is a number between 0 and 1.
            - Order candidates in each list by descending confidence.
            - If there are no candidates, use [].

            Return only the JSON array.
            """
        )
    ]
)

resolver_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            """
            You are choosing between two OWL inverse properties for ONE table column.

            Table: {table_name}
            Column: {column_name}

            Two candidate properties are OWL inverses of each other. Choose the ONE that best matches the column semantics.

            Rules (STRICT):
                - Return ONLY minified JSON (single line).
                - No markdown fences, no extra text.
                - Output schema: {{"winner":"<n>","loser":"<n>"}}
                - winner and loser MUST be exactly one of the two NAMES below.

            Candidates:
                A: {prop_a_name}
                B: {prop_b_name}
            """
        )
    ]
)

ambiguity_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "human",
            """
            You are resolving an ambiguous ontology class name for ONE table column.

            Table: {table_name}
            Column: {column_name}

            The predicted class name "{ambiguous_name}" is ambiguous: it matches multiple ontology classes.
            Below are the candidate classes with their disambiguated names, parent classes, and descriptions.

            Candidates:
            {candidates_text}

            Choose the ONE candidate whose disambiguated name best matches the column semantics.

            Rules (STRICT):
                - Return ONLY minified JSON (single line).
                - No markdown fences, no extra text.
                - Output schema: {{"winner":"<disambiguated_name>"}}
                - winner MUST be exactly one of the disambiguated names listed above.

            Return only the JSON.
            """
        )
    ]
)

ANNOTATION_CHAIN_REGISTRY = {
    "deepseek-chat":    annotation_prompt | llm1,
    "gemini-2.0-flash": annotation_prompt | llm2,
    "gpt-4.1-mini":     annotation_prompt | llm3,
}

RESOLVER_CHAIN_REGISTRY = {
    "deepseek-chat":    resolver_prompt | llm1,
    "gemini-2.0-flash": resolver_prompt | llm2,
    "gpt-4.1-mini":     resolver_prompt | llm3,
}

AMBIGUITY_CHAIN_REGISTRY = {
    "deepseek-chat":    ambiguity_prompt | llm1,
    "gemini-2.0-flash": ambiguity_prompt | llm2,
    "gpt-4.1-mini":     ambiguity_prompt | llm3,
}

In [ ]:
# =========================
# LLM output processing
# =========================
def extract_json_from_response(text):
    """
    Robust JSON extractor from LLM responses.
    """
    m = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    s = m.group(1).strip() if m else text.strip()

    if '\\"' in s:
        try:
            unescaped = json.loads(f'"{s}"')
            if isinstance(unescaped, str):
                s = unescaped.strip()
        except Exception:
            pass

    first_brace   = s.find("{")
    first_bracket = s.find("[")
    starts = [i for i in [first_brace, first_bracket] if i != -1]
    if starts:
        start   = min(starts)
        end_obj = s.rfind("}")
        end_arr = s.rfind("]")
        end     = max(end_obj, end_arr)
        if end != -1 and end > start:
            s = s[start:end + 1].strip()

    def _bracket_depth(ss):
        depth  = 0
        in_str = False
        esc    = False
        for ch in ss:
            if in_str:
                if esc:
                    esc = False
                elif ch == "\\":
                    esc = True
                elif ch == '"':
                    in_str = False
                continue
            else:
                if ch == '"':
                    in_str = True
                    continue
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
        return depth

    if s.startswith("["):
        d = _bracket_depth(s)
        if d == 1 and s.rstrip().endswith("}"):
            s = s.rstrip() + "]"

    return s


def process_llm_output(llm_output_str, table_id, table_name):
    """
    Convert raw LLM JSON output to per-column records.
    No threshold applied here — full candidate list preserved.

    Output record format:
    {
        "table_id": ...,
        "table_name": ...,
        "column_name": ...,
        "column_id": ...,
        "class": [["ClassName", conf], ...],
        "property": [["propName", conf], ...],
        "inverse_conflicts": [],
        "ambiguous_classes": []
    }
    """
    results = []

    try:
        parsed = json.loads(llm_output_str)
        if not isinstance(parsed, list):
            parsed = []
    except json.JSONDecodeError as e:
        print(f"JSON fail on {table_id}: {e}")
        parsed = []

    for i, col in enumerate(parsed):
        if not isinstance(col, dict):
            continue

        column_name = col.get("column", f"Column_{i}")

        class_out = []
        for it in col.get("class", []) or []:
            if not (isinstance(it, (list, tuple)) and len(it) >= 2):
                continue
            label, conf = it[0], it[1]
            if not isinstance(label, str) or not label:
                continue
            try:
                conf = float(conf)
            except Exception:
                continue
            conf = max(0.0, min(1.0, round(conf, 4)))
            class_out.append([label, conf])

        prop_out = []
        for it in col.get("property", []) or []:
            if not (isinstance(it, (list, tuple)) and len(it) >= 2):
                continue
            label, conf = it[0], it[1]
            if not isinstance(label, str) or not label:
                continue
            try:
                conf = float(conf)
            except Exception:
                continue
            conf = max(0.0, min(1.0, round(conf, 4)))
            prop_out.append([label, conf])

        results.append({
            "table_id":          table_id,
            "table_name":        table_name,
            "column_name":       column_name,
            "column_id":         i,
            "class":             class_out,
            "property":          prop_out,
            "inverse_conflicts": [],
            "ambiguous_classes": [],
        })

    return results


def resolve_ambiguous_classes(
    filtered_results,
    table_name,
    ambiguity_chain,
    ambiguous_class_map,
):
    """
    Detect and resolve ambiguous class predictions.

    For each column, scan class predictions:
    - If predicted name matches a base_label in ambiguous_class_map
      (i.e. LLM output the raw ambiguous name instead of Building1/Building2),
      call ambiguity_chain to pick the correct disambiguated name.
    - Replace the ambiguous name with the resolved name.
    - Record the resolution in ambiguous_classes field.
    """
    logs = []

    # build valid disambiguated names set for quick lookup
    valid_disamb_names = set()
    for candidates in ambiguous_class_map.values():
        for c in candidates:
            valid_disamb_names.add(c["name"])

    for col in filtered_results:
        new_class_list  = []
        ambiguous_records = []

        for label, conf in col.get("class", []):
            # check if this is a raw ambiguous name
            if label in ambiguous_class_map:
                candidates = ambiguous_class_map[label]

                # build candidates text for prompt
                candidates_text = "\n".join([
                    f"  - Disambiguated name: {c['name']}\n"
                    f"    Parents: {', '.join(c.get('parents', [])) or 'N/A'}\n"
                    f"    Description: {c.get('comment', '') or 'N/A'}"
                    for c in candidates
                ])

                winner = None
                raw    = ""
                err    = ""

                try:
                    resp = ambiguity_chain.invoke({
                        "table_name":      table_name,
                        "column_name":     col["column_name"],
                        "ambiguous_name":  label,
                        "candidates_text": candidates_text,
                    })
                    raw = getattr(resp, "content", "")
                    obj = json.loads(extract_json_from_response(raw))
                    w   = obj.get("winner", "").strip()
                    if w in valid_disamb_names:
                        winner = w
                    else:
                        err = f"invalid_winner: {w}"
                except Exception as e:
                    err = f"error: {e}"

                record = {
                    "column_name":     col["column_name"],
                    "column_id":       col["column_id"],
                    "ambiguous_name":  label,
                    "confidence":      conf,
                    "candidates":      candidates,
                    "resolved_to":     winner,
                    "raw":             raw,
                    "error":           err,
                }
                ambiguous_records.append(record)
                logs.append(record)

                if winner:
                    new_class_list.append([winner, conf])
                # if resolution failed, drop the ambiguous prediction

            elif label in valid_disamb_names:
                # LLM correctly output a disambiguated name, keep as-is
                new_class_list.append([label, conf])
            else:
                # normal unambiguous name
                new_class_list.append([label, conf])

        col["class"]             = new_class_list
        col["ambiguous_classes"] = ambiguous_records

    return filtered_results, logs


def invoke_with_retry(chain, input_dict, max_retries=3, wait=10):
    for attempt in range(max_retries):
        try:
            return chain.invoke(input_dict)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                print(f"  Rate limit hit, waiting {wait}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Failed after {max_retries} retries")

In [ ]:
# =========================
# Single experiment runner
# =========================
def run_single_experiment(model_names):
    chains            = [ANNOTATION_CHAIN_REGISTRY[name] for name in model_names]
    resolver_chains   = [RESOLVER_CHAIN_REGISTRY[name]   for name in model_names]
    ambiguity_chains  = [AMBIGUITY_CHAIN_REGISTRY[name]  for name in model_names]
    num_agents        = len(model_names)

    candidate_file_path = build_candidate_path(model_names)
    logs_path           = build_logs_path(model_names)
    summary_path        = build_summary_path(model_names)

    print("=" * 80)
    print(f"Running experiment: {build_experiment_tag(model_names)}")
    print("Models:            ", model_names)
    print("Candidate file:    ", candidate_file_path)
    print("=" * 80)

    start_time = time.perf_counter()

    all_llm_records   = []
    candidate_results = []

    for table_id in df_table_list["table_id"].unique():
        table             = dict_tables[table_id]
        table_name        = df_table_list[df_table_list["table_id"] == table_id]["table_name"].values[0]
        table_in_markdown = dataframe_to_markdown(table, k)

        print("+" * 70)
        print(table_id, table_name)

        input_dict = {
            "table_name":       table_name,
            "table_in_markdown": table_in_markdown,
            "ontology_json":    ontology_json,
        }

        for agent_id in range(num_agents):
            model_name       = model_names[agent_id]
            chain            = chains[agent_id]
            resolver_chain   = resolver_chains[agent_id]
            ambiguity_chain  = ambiguity_chains[agent_id]

            # ── prompt logging ────────────────────────────────────────────────
            prompt_template  = chain.first
            formatted_prompt = prompt_template.format(**input_dict)
            if isinstance(formatted_prompt, str):
                prompt_text = formatted_prompt.strip()
            else:
                messages    = formatted_prompt.to_messages()
                prompt_text = "\n".join([f"[{m.type.upper()}]\n{m.content}" for m in messages])

            # ── LLM inference ─────────────────────────────────────────────────
            response = invoke_with_retry(chain, input_dict)
            time.sleep(1)
            output_text = extract_json_from_response(response.content)

            usage_raw   = response.response_metadata.get("token_usage", {})
            usage_clean = {
                "input_tokens":  usage_raw.get("prompt_tokens", 0),
                "output_tokens": usage_raw.get("completion_tokens", 0),
                "total_tokens":  usage_raw.get("total_tokens", 0),
            }

            all_llm_records.append({
                "table_id":   table_id,
                "table_name": table_name,
                "model":      model_name,
                "prompt":     prompt_text,
                "output":     response.content,
                "usage":      usage_clean,
            })

            # ── process output ────────────────────────────────────────────────
            filtered_results = process_llm_output(
                llm_output_str=output_text,
                table_id=table_id,
                table_name=table_name,
            )

            # ── Step 1: ambiguity resolution ──────────────────────────────────
            filtered_results, amb_logs = resolve_ambiguous_classes(
                filtered_results=filtered_results,
                table_name=table_name,
                ambiguity_chain=ambiguity_chain,
                ambiguous_class_map=AMBIGUOUS_CLASS_MAP,
            )

            # ── Step 2: inverse conflict detection + resolution ───────────────
            filtered_results, inv_logs = inv_mgr.fix_after_process_llm_output(
                filtered_results=filtered_results,
                table_name=table_name,
                resolver_chain=resolver_chain,
                extract_json_from_response=extract_json_from_response,
            )

            # ── store candidate ───────────────────────────────────────────────
            for item in filtered_results:
                candidate_results.append({
                    "table_id":          item["table_id"],
                    "table_name":        item["table_name"],
                    "model":             model_name,
                    "column_name":       item["column_name"],
                    "column_id":         item["column_id"],
                    "class":             item["class"],
                    "property":          item["property"],
                    "inverse_conflicts": item.get("inverse_conflicts", []),
                    "ambiguous_classes": item.get("ambiguous_classes", []),
                })

            print(f"  [{model_name}] done")

    # ── timekeeping ───────────────────────────────────────────────────────────
    end_time        = time.perf_counter()
    elapsed_time    = end_time - start_time
    elapsed_minutes = elapsed_time / 60

    # ── token summary ─────────────────────────────────────────────────────────
    token_summary = {
        "total":    {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "by_model": defaultdict(lambda: {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}),
    }
    for rec in all_llm_records:
        model = rec.get("model", "unknown_model")
        usage = rec.get("usage", {})
        for key in ["input_tokens", "output_tokens", "total_tokens"]:
            value = usage.get(key, 0) or 0
            token_summary["total"][key] += value
            token_summary["by_model"][model][key] += value

    summary = {
        "experiment": build_experiment_tag(model_names),
        "models":     model_names,
        "run_time": {
            "seconds": round(elapsed_time, 2),
            "minutes": round(elapsed_minutes, 2),
        },
        "token_usage_summary": {
            "total":    token_summary["total"],
            "by_model": dict(token_summary["by_model"]),
        },
    }

    with open(candidate_file_path, "w", encoding="utf-8") as f:
        json.dump(candidate_results, f, ensure_ascii=False, indent=2)

    with open(logs_path, "w", encoding="utf-8") as f:
        json.dump(all_llm_records, f, ensure_ascii=False, indent=2)

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"Saved candidate -> {candidate_file_path}")
    print(f"Saved logs      -> {logs_path}")
    print(f"Saved summary   -> {summary_path}")

In [ ]:
def run_all_experiments():
    for model_names in EXPERIMENTS:
        run_single_experiment(model_names=model_names)

run_all_experiments()